# Week 7: cleaning and refactoring the assignment notebook

This notebook keeps the original analysis goal, but makes the code easier to maintain.

It addresses the assignment by:
- removing repeated code with loops,
- wrapping repeated logic into functions,
- replacing hardcoded paths, sheets, countries, and task blocks with configuration.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SELECTED_COUNTRIES = ["Belgium", "Poland", "Spain"]
SELECTED_TASK_GROUP = "NRCA"
TASK_GROUPS = {
    "NRCA": ["t_4A2a4", "t_4A2b2", "t_4A4a1"],
}

In [ ]:
def find_project_root(start: Path) -> Path:
    """Walk up from the current working directory until the project Data folder is found."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing a Data directory.")


def validate_columns(frame: pd.DataFrame, columns: list[str], label: str) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"Missing {label}: {missing}")


def load_employment_data(excel_path: Path) -> pd.DataFrame:
    """Read every ISCO sheet from the Eurostat workbook and stack them into one table."""
    workbook = pd.ExcelFile(excel_path)
    frames = []

    for sheet_name in workbook.sheet_names:
        if not sheet_name.startswith("ISCO"):
            continue

        isco_code = int(sheet_name.removeprefix("ISCO"))
        sheet_data = pd.read_excel(workbook, sheet_name=sheet_name)
        sheet_data["ISCO"] = isco_code
        frames.append(sheet_data)

    return pd.concat(frames, ignore_index=True)


def add_country_shares(frame: pd.DataFrame, countries: list[str]) -> pd.DataFrame:
    """Create total-employment and employment-share columns for each selected country."""
    enriched = frame.copy()

    for country in countries:
        total_column = f"total_{country}"
        share_column = f"share_{country}"
        enriched[total_column] = enriched.groupby("TIME")[country].transform("sum")
        enriched[share_column] = enriched[country] / enriched[total_column]

    return enriched


def prepare_task_data(task_path: Path) -> pd.DataFrame:
    """Aggregate O*NET task scores from detailed occupations to the 1-digit ISCO level."""
    task_data = pd.read_csv(task_path)
    task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[0].astype(int)
    return task_data.groupby("isco08_1dig").mean(numeric_only=True).reset_index()


def weighted_standardize(values: pd.Series, weights: pd.Series) -> pd.Series:
    """Standardize a series using weighted mean and weighted standard deviation."""
    mask = values.notna() & weights.notna()
    result = pd.Series(np.nan, index=values.index, dtype=float)

    valid_values = values[mask]
    valid_weights = weights[mask]
    weighted_mean = np.average(valid_values, weights=valid_weights)
    weighted_sd = np.sqrt(np.average((valid_values - weighted_mean) ** 2, weights=valid_weights))

    result.loc[mask] = (valid_values - weighted_mean) / weighted_sd
    return result


def add_standardized_tasks(frame: pd.DataFrame, countries: list[str], task_columns: list[str]) -> pd.DataFrame:
    """Standardize each selected task separately for every selected country."""
    enriched = frame.copy()

    for task_column in task_columns:
        for country in countries:
            share_column = f"share_{country}"
            standardized_column = f"std_{country}_{task_column}"
            enriched[standardized_column] = weighted_standardize(enriched[task_column], enriched[share_column])

    return enriched


def build_task_index(
    frame: pd.DataFrame,
    countries: list[str],
    task_group_name: str,
    task_columns: list[str],
) -> pd.DataFrame:
    """Create a composite task index, standardize it, and weight it by country employment shares."""
    enriched = add_standardized_tasks(frame, countries, task_columns)

    for country in countries:
        standardized_task_columns = [f"std_{country}_{task_column}" for task_column in task_columns]
        raw_index_column = f"{country}_{task_group_name}"
        standardized_index_column = f"std_{country}_{task_group_name}"
        weighted_index_column = f"weighted_{country}_{task_group_name}"
        share_column = f"share_{country}"

        enriched[raw_index_column] = enriched[standardized_task_columns].sum(axis=1)
        enriched[standardized_index_column] = weighted_standardize(enriched[raw_index_column], enriched[share_column])
        enriched[weighted_index_column] = enriched[standardized_index_column] * enriched[share_column]

    return enriched


def summarize_country_series(frame: pd.DataFrame, countries: list[str], task_group_name: str) -> pd.DataFrame:
    """Collapse the weighted task index to one time series per country."""
    series_list = []

    for country in countries:
        weighted_index_column = f"weighted_{country}_{task_group_name}"
        country_series = (
            frame.groupby("TIME", as_index=False)[weighted_index_column]
            .sum()
            .rename(columns={weighted_index_column: country})
        )
        series_list.append(country_series)

    result = series_list[0]
    for additional_series in series_list[1:]:
        result = result.merge(additional_series, on="TIME", how="left")

    return result


def plot_country_series(series: pd.DataFrame, countries: list[str], task_group_name: str) -> None:
    """Plot the selected task index over time for each selected country."""
    fig, ax = plt.subplots(figsize=(12, 6))

    for country in countries:
        ax.plot(series["TIME"], series[country], label=country, linewidth=2)

    tick_positions = list(range(0, len(series), 3))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(series["TIME"].iloc[tick_positions], rotation=45, ha="right")
    ax.set_title(f"{task_group_name} task intensity over time")
    ax.set_xlabel("Quarter")
    ax.set_ylabel("Weighted standardized index")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
project_root = find_project_root(Path.cwd())
data_dir = project_root / "Data"

employment_data = load_employment_data(data_dir / "Eurostat_employment_isco.xlsx")
validate_columns(employment_data, ["TIME", *SELECTED_COUNTRIES], "country columns")
employment_data = add_country_shares(employment_data, SELECTED_COUNTRIES)

task_data = prepare_task_data(data_dir / "onet_tasks.csv")
validate_columns(task_data, TASK_GROUPS[SELECTED_TASK_GROUP], "task columns")

combined = employment_data.merge(
    task_data,
    left_on="ISCO",
    right_on="isco08_1dig",
    how="left",
)

combined = build_task_index(
    frame=combined,
    countries=SELECTED_COUNTRIES,
    task_group_name=SELECTED_TASK_GROUP,
    task_columns=TASK_GROUPS[SELECTED_TASK_GROUP],
)

country_series = summarize_country_series(
    frame=combined,
    countries=SELECTED_COUNTRIES,
    task_group_name=SELECTED_TASK_GROUP,
)

display(country_series.head())

In [ ]:
plot_country_series(
    series=country_series,
    countries=SELECTED_COUNTRIES,
    task_group_name=SELECTED_TASK_GROUP,
)

To extend the analysis, update `SELECTED_COUNTRIES` or add another entry to `TASK_GROUPS`.
The rest of the notebook will reuse the same functions without repeating the code.